# Retail Data Analytics with PySpark

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

## Load the table into PySpark DataFrame

In [0]:
retail_df = spark.table("databrick_sara.default.online_retail")

In [0]:
# Checking statistical results
display(retail_df.describe())

summary,Invoice,StockCode,Description,Quantity,Price,CustomerID,Country
count,1067371,1067371,1062989,1067371,1067371,824364,1067371
mean,537608.1499316233,29011.161534536903,21848.25,9.9388984711033,4.649388,15324.6385,null
stddev,26662.45044690566,18822.94286618916,922.9197780233488,172.705794076754,123.55305872209716,1697.4644503793122,null
min,489434,10002,DOORMAT UNION JACK GUNS AND ROSES,-80995,-53594.36,12346,Australia
max,C581569,m,wrongly sold sets,80995,38970.00,18287,West Indies


## Total Invoice Amount Distribution

---
1. Calculate the invoice amount. An invoice consists of one or more items where each item is a row in the df.
2. Draw the distribution of invoice amount with min, max, median, mod, and mean. Plot hist and box charts.
3. Draw the distribution for the first 85 quantiles of the invoice amount data (remove outlier data) with min, max, median, mod, and mean. Plot hist and box charts.

---

In [0]:
invoice_totals = (
    retail_df
    .withColumn("invoice_totals", F.col("Quantity") * F.col("Price"))
    .groupBy("Invoice")
    .agg(F.sum("invoice_totals").alias("invoice_totals"))
)

In [0]:
# Filter out invoices of cancelled items
invoice_totals_filtered = invoice_totals.filter(F.col("invoice_totals") > 0)

# show summary stats
display(invoice_totals_filtered.describe())

summary,Invoice,invoice_totals
count,40078,40078
mean,536213.9320540972,523.303761
stddev,26509.104733599,1517.3516455469835
min,489434,0.19
max,C496350,168469.60


In [0]:
# Filter to the 85th percentile of invoice_total
q85 = invoice_totals_filtered.approxQuantile("invoice_totals", [0.85], 0.01)[0]

invoice_totals_85q = invoice_totals_filtered.filter(F.col("invoice_totals") <= q85)

display(invoice_totals_85q.describe())

summary,Invoice,invoice_totals
count,33699,33699
mean,536164.2275209212,266.920401
stddev,26550.641192200983,169.8502284368228
min,489434,0.19
max,C496350,692.96


## Monthly Placed and Canceled Orders

---

1. To count canceled orders: `invoice` is nominal 6-digit integral number uniquely assigned to each transaction. If this code starts with the letter 'c', it indicates a cancellation. 
2. To calculate placed orders: we assumed that there are two invoice numbers for each canceled order (one for the original invoice and one for the canceled invoice). Therefore, `# of placed orders = total # of orders - 2 * canceled order`. 
3. To simplify the problem, we also assumed the original invoice and canceled invoice are on always on the same day (this eliminate the case where the original invoice and canceled invoices are on different months). We also treat `Adjust bad debt` as placed orders. A record described as `Adjust bad debt` means the company is writing off unpaid customer debts and subtracting them from revenue.

---

In [0]:
# Add year-month column
retail_df = retail_df.withColumn(
    "invoice_year_month",
    F.date_format("InvoiceDate", "yyyyMM")
)

In [0]:
canceled_df = retail_df.filter(F.col("Invoice").startswith("C"))

In [0]:
monthly_canceled_orders = (
    canceled_df.groupBy("invoice_year_month")
    .agg(F.countDistinct("Invoice").alias("Cancellation"))
)

In [0]:
monthly_total_orders = (
    retail_df.groupBy("invoice_year_month")
    .agg(F.countDistinct("Invoice").alias("Total"))
)

In [0]:
monthly_orders = (
    monthly_total_orders
    .join(monthly_canceled_orders, on="invoice_year_month", how="left")
    .withColumn("Placement", F.col("Total") - 2 * F.col("Cancellation"))
)

In [0]:
df = monthly_orders.select("invoice_year_month", "Placement", "Cancellation").orderBy("invoice_year_month")

# call an action to execute
display(df)

invoice_year_month,Placement,Cancellation
200912,1528,401
201001,1033,300
201002,1489,240
201003,1553,407
201004,1284,304
201005,1604,407
201006,1502,357
201007,1329,344
201008,1331,273
201009,1633,371


## Monthly Sales

---

1. Calculate the monthly sales data
2. Plot a chart to show monthly sales

---

In [0]:
# Filter valid transactions
sales_df = retail_df.filter(
    (F.col("Quantity") > 0) & (F.col("Price") > 0)
)
# Compute sales_amount and group by month
monthly_sales_df = (
    sales_df
    .withColumn("SalesAmount", F.col("Quantity") * F.col("Price"))
    .groupBy('invoice_year_month')
    .agg(F.sum("SalesAmount").alias("MonthlySales"))
    .orderBy("invoice_year_month")
)
# Call an action
display(monthly_sales_df)

invoice_year_month,MonthlySales
200912,825685.76
201001,652708.50
201002,553713.30
201003,833570.13
201004,681528.99
201005,659858.86
201006,752270.14
201007,650712.94
201008,697274.91
201009,924333.01


## Monthly Sales Growth


---

1. Calculate monthly sales percentage growth data using `pandas.Series.pct_change()`
2. Plot a chart to show the growth percentage

---

In [0]:
# Define window ordered by invoice_year_month
w = Window.orderBy("invoice_year_month")

# Add previous month sales using lag()
monthly_sales_df = monthly_sales_df.withColumn(
    "PrevSales",
    F.lag("MonthlySales").over(w)
)

# Calculate sales_growth = (current - previous) / previous
monthly_sales_df = monthly_sales_df.withColumn(
    "SalesGrowth",
    F.when(F.col("PrevSales").isNotNull(),
           (F.col("MonthlySales") - F.col("PrevSales")) / F.col("PrevSales"))
     .otherwise(None)
)

# Drop prev_sales
monthly_sales_df = monthly_sales_df.drop("PrevSales")

# Step 4: Display result
display(monthly_sales_df)


invoice_year_month,MonthlySales,SalesGrowth
200912,825685.76,null
201001,652708.50,-0.209495
201002,553713.30,-0.151668
201003,833570.13,0.505418
201004,681528.99,-0.182398
201005,659858.86,-0.031796
201006,752270.14,0.140047
201007,650712.94,-0.135001
201008,697274.91,0.071555
201009,924333.01,0.325636


## Monthly Active Users

---

1. Compute # of active users (e.g. unique `CusotomerID`) for each month
2. Plot a bar chart

---

In [0]:
monthly_active_users_df = (
    retail_df
    .groupby('invoice_year_month')
    .agg(F.countDistinct("CustomerID").alias("MonthlyActiveUsers"))
    .orderBy("invoice_year_month")
)
display(monthly_active_users_df)

invoice_year_month,MonthlyActiveUsers
200912,1045
201001,786
201002,807
201003,1111
201004,998
201005,1062
201006,1095
201007,988
201008,964
201009,1202


## New and Existing Users



---

1. Find out the first purchase year-month for each user and then join this data with the transactional data to identify new/exiting users

    - A user is identified as a new user when he/she makes the first purchase
    - A user is identified as an existing user when he/she made purchases in the past

2. Plot a diagram to show new and exiting user for each month.

---

In [0]:
# Get first purchase month for each customer
first_purchase = (
    retail_df
    .groupBy('CustomerID')
    .agg(F.min("invoice_year_month").alias("FirstPurchaseMonth"))
)

In [0]:
# Classify each invoice as from a NewUser or ExUser
retail_df_with_user_type = (
    retail_df
    .join(first_purchase, on="CustomerID", how="left")
    .withColumn(
        "UserType",
        F.when(F.col("invoice_year_month") == F.col("FirstPurchaseMonth"), "NewUser")
        .otherwise("ExUser")
    ))

In [0]:
# Group by year-month and user type, count distinct customers
grouped_user_counts = (
    retail_df_with_user_type
    .groupBy("invoice_year_month", "UserType")
    .agg(F.countDistinct("CustomerID").alias("UserCounts"))
)

In [0]:
monthly_user_type = (
    grouped_user_counts
    .groupBy("invoice_year_month")
    .pivot("UserType", ["NewUser", "ExUser"])
    .agg(F.sum("UserCounts"))
    .fillna(0)
    .orderBy("invoice_year_month")
)
display(monthly_user_type)

invoice_year_month,NewUser,ExUser
200912,1045,0
201001,394,392
201002,363,444
201003,436,675
201004,291,707
201005,254,808
201006,269,826
201007,183,805
201008,158,806
201009,242,960


## RFM Segmentation

RFM is a method used for analyzing customer value. It is commonly used in database marketing and direct marketing and has received particular attention in the retail and professional services industries. ([wikipedia](https://en.wikipedia.org/wiki/RFM_(market_research)))

RFM stands for three dimensions:

- Recency: How recently did the customer purchase?

- Frequency: How often do they purchase?

- Monetary Value: How much do they spend?

To simplify the problem, we used order data with positive quantities and price for evaluation. 


In [0]:
# to make the assessment easier, reference date is set as January 1, 2012.  
reference_date = F.lit("2012-01-01")

In [0]:
cleaned_retail_df = (
    retail_df
    .filter((F.col("Quantity") > 0) & (F.col("Price") > 0))
    .withColumn("TotalAmount", F.col("Quantity") * F.col("Price"))
    .dropna()
)
display(cleaned_retail_df.describe())

summary,Invoice,StockCode,Description,Quantity,Price,CustomerID,Country,TotalAmount
count,805531,805531,805531,805531,805531,805531,805531,805531
mean,537411.1437039668,28804.292780619366,null,13.290797002225862,3.206633,15331.9710,null,22.026997
stddev,26665.922214763374,18501.96201499669,null,143.63568136266574,29.199494957263866,1696.7417854594448,null,224.0444065009785
min,489434,10002,DOORMAT UNION JACK GUNS AND ROSES,1,0.03,12346,Australia,0.06
max,581587,TEST002,ZINC WIRE SWEETHEART LETTER TRAY,80995,10953.50,18287,West Indies,168469.60


In [0]:
recency_df = (
    cleaned_retail_df
    .groupBy("CustomerID")
    .agg(F.max("InvoiceDate").alias("LastPurchaseDate"))
    .withColumn("Recency", F.datediff(F.to_date(reference_date), F.to_date("LastPurchaseDate")))
)


In [0]:
frequency_df = (
    cleaned_retail_df
    .groupBy("CustomerID")
    .agg(F.countDistinct("Invoice").alias("Frequency"))
)

In [0]:
monetary_df = (
    cleaned_retail_df
    .groupBy("CustomerID")
    .agg(F.sum("TotalAmount").alias("Monetary"))
)

In [0]:
rfm = (
    recency_df
    .join(frequency_df, on="CustomerID")
    .join(monetary_df, on="CustomerID")
    .drop("LastPurchaseDate")
    .orderBy("CustomerID")
)
display(rfm)

CustomerID,Recency,Frequency,Monetary
12346,348,12,77556.46
12347,25,8,5633.32
12348,98,5,2019.40
12349,41,4,4428.69
12350,333,1,334.40
12351,398,1,300.93
12352,59,10,2849.84
12353,227,2,406.76
12354,255,1,1079.40
12355,237,2,947.61


## RFM Segmentation

---
RFM segmentation categorizes customers into different segments, according to their interactions with the company's website, which will help subsequently approach these groups in the most effective way. 

According to the following articles, we can make an RFM segmentation based on an RFM score combining all three RFM parameters together and allowing us to divide our customers into 11 different segments. 

- [RFM Segmentation business cases](https://docs.exponea.com/docs/rfm-segmentation-business-use)
- [RFM Segmentation Guide](https://docs.exponea.com/docs/rfm-segmentation-business-use)

In this report, we used only Recency and Frequency for RFM Segmentation:

- Focus on Customer Behavior: Recency and Frequency capture behavioral patterns - "how recently a customer purchased" & "how often they purchase" - these directly reflect customer engagement and loyalty.

- Simplifies Segmentation: Using only Recency and Frequency, we create 25 combinations (5 × 5). However, including Monetary would result in 125 combinations (5 × 5 × 5), making segmentation too granular and harder to interpret.
---

In [0]:
# Windows ordered by ascending/descending metric
recency_window = Window.orderBy("Recency")
frequency_window = Window.orderBy(F.col("Frequency").desc())
monetary_window = Window.orderBy(F.col("Monetary").desc())

# Assign scores: 1 (lowest) to 5 (highest)
rfm = (
    rfm
    .withColumn("RecencyScore", F.ntile(5).over(recency_window))
    .withColumn("FrequencyScore", F.ntile(5).over(frequency_window))
    .withColumn("MonetaryScore", F.ntile(5).over(monetary_window))
)

In [0]:
#calculation of the RFM score
rfm = rfm.withColumn(
    "RFM_SCORE",
    F.concat_ws("", F.col("RecencyScore"), F.col("FrequencyScore"), F.col("MonetaryScore"))
)


In [0]:
# 10 segments of customers according to RecencyScore and FrequencyScore values
rfm = rfm.withColumn(
    "SegmentCode",
    F.concat_ws("", F.col("RecencyScore"), F.col("FrequencyScore"))
)

rfm = rfm.withColumn(
    "Segment",
    F.when(F.col("SegmentCode").rlike(r"^[1-2][1-2]$"), "Hibernating")
     .when(F.col("SegmentCode").rlike(r"^[1-2][3-4]$"), "At Risk")
     .when(F.col("SegmentCode") == "15", "Can't Lose")
     .when(F.col("SegmentCode").rlike(r"^3[1-2]$"), "About to Sleep")
     .when(F.col("SegmentCode") == "33", "Need Attention")
     .when(F.col("SegmentCode").rlike(r"^[3-4][4-5]$"), "Loyal Customers")
     .when(F.col("SegmentCode") == "41", "Promising")
     .when(F.col("SegmentCode") == "51", "New Customers")
     .when(F.col("SegmentCode").rlike(r"^[4-5][2-3]$"), "Potential Loyalists")
     .when(F.col("SegmentCode").rlike(r"^5[4-5]$"), "Champions")
     .otherwise("Others")
)


In [0]:
segment_df = (
    rfm
    .groupBy("Segment")
    .agg(
        F.round(F.avg("Recency"), 0).alias("AvgRecency"),
        F.round(F.avg("Frequency"), 0).alias("AvgFrequency"),
        F.round(F.avg("Monetary"), 0).alias("AvgMonetary"),
        F.count("*").alias("CustomerCount")
    )
    .orderBy("CustomerCount")
)

display(segment_df)


Segment,AvgRecency,AvgFrequency,AvgMonetary,CustomerCount
New Customers,491.0,17.0,6589,16
Promising,307.0,17.0,9783,48
Need Attention,136.0,3.0,1365,277
About to Sleep,127.0,9.0,3535,480
Potential Loyalists,387.0,4.0,1537,642
At Risk,49.0,2.0,933,763
Champions,579.0,1.0,441,952
Loyal Customers,261.0,1.0,491,1111
Hibernating,43.0,15.0,7823,1589
